# 第 8 章:预训练 —— 下一 token 预测循环

从本章开始,我们进入教程的第二部分:**训练**。前 7 章我们搭好了模型(ch2-7),现在要让它的权重从随机变成「能理解语言」。

预训练是所有后续阶段(SFT、LoRA、对齐)的基础。本章拆解 minimind 的预训练循环 —— 这个循环也是后面所有训练脚本(SFT/LoRA/DPO/PPO)共用的骨架。

## 8.1 预训练的目标

预训练让模型学会**语言的统计规律**:给定前面的 token,预测下一个 token 最可能是什么。

```
输入:  「今天天气真」
目标:  「好」
```

这是**自监督学习**:不需要人工标注,文本本身就是标签(shift by 1)。

损失函数:交叉熵(Cross-Entropy)

$$\mathcal{L} = -\sum_{t=1}^{T} \log P(x_t | x_{<t})$$

minimind 默认配置:`batch_size=32`, `lr=5e-4`, `epochs=2`, `max_seq_len=340`。

## 8.2 PretrainDataset

读 `lm_dataset.py:37-55`:

```python
class PretrainDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_length=340):
        self.data = load_dataset('json', data_files=data_path)['train']
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __getitem__(self, idx):
        text = self.data[idx]['text']
        # 文本 → token id
        token_ids = self.tokenizer.encode(text)
        # 添加 bos 和 eos
        token_ids = [bos_id] + token_ids + [eos_id]
        # 截断或填充
        token_ids = token_ids[:self.max_length]
        token_ids += [pad_id] * (self.max_length - len(token_ids))
        # 构造 labels(input_ids 的副本,pad 位置设为 -100)
        labels = token_ids.copy()
        labels = [-100 if t == pad_id else t for t in labels]
        return {'input_ids': token_ids, 'labels': labels}
```

> **为什么 label 的 pad 位置设为 -100?** `F.cross_entropy` 的 `ignore_index=-100` 会跳过这些位置。pad token 不应该参与 loss 计算 —— 否则模型会学着「生成 padding」,这不是我们想要的。

## 8.3 训练骨架(8 个脚本共用的模板)

minimind 的 8 个训练脚本(pretrain/SFT/LoRA/DPO/蒸馏/PPO/GRPO/Agent)共用一套骨架:

```
init_distributed_mode()        # ① DDP 初始化
config = MiniMindConfig(...)   # ② 模型配置
model = init_model(...)         # ③ 加载模型(可选 from_weight)
dataset = XxxDataset(...)       # ④ 数据集
optimizer = AdamW(...)          # ⑤ 优化器
for epoch:                      # ⑥ 训练循环
    for batch:
        with autocast():        #    混合精度
            loss = model(batch)
        scaler.scale(loss).backward()
        clip_grad_norm_(model, 1.0)  # 梯度裁剪
        scaler.step(optimizer)
    lm_checkpoint(...)          #    保存 checkpoint
```

> **关键设计**:每个训练脚本只是在这个骨架上替换 ④(数据集类)和 loss 的计算方式。理解了预训练循环,就理解了全部 8 个脚本的结构。

## 8.4 余弦学习率调度

读 `get_lr` — trainer_utils.py:~40 (@67f114a):

$$\text{lr}(t) = \text{lr}_{\text{base}} \cdot \left(0.1 + 0.45 \cdot \left(1 + \cos\left(\frac{\pi \cdot t}{T}\right)\right)\right)$$

```python
def get_lr(step, total_steps, base_lr):
    # 前 10%: 爬坡(warmup)
    # 中间: 余弦下降
    # 公式确保 lr 在底部(10%×base)起步,峰值后余弦衰减回底部
    return base_lr * (0.1 + 0.45 * (1 + math.cos(math.pi * step / total_steps)))
```

三个阶段:
- **开始(step≈0)**:lr ≈ base × (0.1 + 0.45 × 2) = base × 1.0 → 峰值
- **中间(step=T/2)**:lr ≈ base × (0.1 + 0.45 × 1) = base × 0.55
- **结束(step=T)**:lr ≈ base × (0.1 + 0.45 × 0) = base × 0.1 → 底部

> **为什么不用固定 lr?** 固定 lr 容易在训练后期震荡。余弦调度让 lr 逐渐降低,模型在后期做更精细的调整。

## 8.5 AdamW 优化器

```python
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-4,           # 预训练 lr(比 SFT 的 1e-5 高 50 倍)
    weight_decay=0.1,  # decoupled weight decay
)
# 梯度裁剪
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
```

**为什么 AdamW 不用 Adam?** AdamW 的 decoupled weight decay 比 Adam 的 L2 正则化更有效 —— 它直接缩小权重,而不是通过梯度。这对大模型训练很重要。

> **lr=5e-4** 是预训练的学习率。注意 SFT 的 lr 只有 1e-5(低 50 倍!)—— 因为预训练从零开始需要大步学习,而 SFT 是微调已有权重,小步调整即可。

## 8.6 混合精度训练(AMP)

```python
from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler()
for batch in dataloader:
    optimizer.zero_grad()
    with autocast(dtype=torch.bfloat16):  # 前向用 bfloat16
        loss = model(batch)
    scaler.scale(loss).backward()   # 反向用 float32(自动)
    scaler.step(optimizer)          # 更新
    scaler.update()
```

| 精度 | 显存 | 速度 | 稳定性 |
|---|---|---|---|
| float32 | 2x | 基准 | 最稳定 |
| float16 | 1x | ~2x | 可能溢出 |
| **bfloat16** | **1x** | **~2x** | **稳定(指数位多)** |

> minimind 用 bfloat16 —— 比 float16 更稳定(指数位与 float32 相同),现代 GPU(3090+)原生支持。

## 8.7 DDP 分布式训练

```python
def init_distributed_mode():
    torch.distributed.init_process_group(backend='nccl')
    model = DDP(model, device_ids=[local_rank])
```

DDP(Distributed Data Parallel)的核心:
- 每张 GPU 有完整的模型副本
- 数据被分到各 GPU(每卡处理不同 batch)
- 反向传播后,**梯度自动同步**(all-reduce 平均)

> minimind 支持单机多卡。单卡也能跑,只是慢。目标是:单张 3090,2 小时跑完 1 epoch SFT。

## 8.8 Checkpoint 与断点续训

读 `lm_checkpoint` — trainer_utils.py:~71-125 (@67f114a):

master 把 checkpoint 的**权重后缀逻辑**抽成了一个共用函数 `_model_suffix`(`trainer_utils.py:~63-70`),让全部 6 个训练脚本(pretrain/SFT/LoRA/DPO/PPO/GRPO)都走同一条路径拼接:

```python
def _model_suffix(lm_config):
    # 权重后缀: MoE / PLE / Dense 各自独立, 避免互相覆盖
    if getattr(lm_config, 'use_ple', False):
        return '_ple'
    if getattr(lm_config, 'use_moe', False):
        return '_moe'
    return ''   # 默认 Dense: 后缀为空

def lm_checkpoint(lm_config, weight='full_sft', model=None, optimizer=None,
                  epoch=0, step=0, wandb=None, save_dir='../checkpoints', **kwargs):
    suffix = _model_suffix(lm_config)
    ckp_path    = f'{save_dir}/{weight}_{lm_config.hidden_size}{suffix}.pth'
    resume_path = f'{save_dir}/{weight}_{lm_config.hidden_size}{suffix}_resume.pth'

    if model is not None:          # 保存模式
        # 半精度 + 原子写(先 .tmp 再 os.replace, 防中途崩溃产生坏文件)
        torch.save(state_dict, ckp_path)
        torch.save(resume_data, resume_path)   # 含 optimizer/epoch/step/world_size
    else:                          # 加载模式: 读 _resume.pth 续训
        return torch.load(resume_path, map_location='cpu')
```

> **默认 Dense 模式 `suffix=''`,checkpoint 路径与上游原版完全一致** —— 预训练产出 `out/pretrain_768.pth`、SFT 产出 `out/full_sft_768.pth`,不插任何后缀。只有 `use_ple` / `use_moe` 打开时才分别加 `_ple` / `_moe`(如 `pretrain_768_ple.pth`),不同架构权重不会互相覆盖。

> **为什么要 `_model_suffix`?** Dense / PLE / MoE 三种 FFN 架构的权重张量形状不同,若共用同一文件名会互相覆盖。`_model_suffix` 让每种架构落盘到独立文件,6 个训练脚本共用一行逻辑,免得在每个脚本里硬编码后缀。

**SkipBatchSampler** — trainer_utils.py:~142-166 (@67f114a):恢复时跳过已训练的 batch,确保数据顺序一致。

> **跨 GPU 数量恢复**:如果用 4 卡训了 1000 步,然后用 2 卡恢复,checkpoint 仍然有效 —— 因为每步的梯度更新是确定的(与 GPU 数量无关,DDP 只影响每步处理多少数据)。

## 8.9 完整训练命令

```bash
cd trainer
python train_pretrain.py \
    --epochs 2 \
    --batch_size 32 \
    --accumulation_steps 8 \  # 梯度累积(等效 batch=256)
    --lr 5e-4 \
    --max_seq_len 340 \
    --data_path ../dataset/pretrain_t2t_mini.jsonl \
    --from_weight none  # 从零开始
```

输出:`out/pretrain_768.pth`(约 250MB)

> **架构后缀**:默认 Dense 输出 `out/pretrain_768.pth`(后缀为空)。加 `--use_moe 1` → `pretrain_768_moe.pth`(loss 自动加 `aux_loss`);加 `--use_ple 1` → `pretrain_768_ple.pth`。后缀由 `_model_suffix` 决定(见 §8.8)。

&nbsp;

---

## Summary and takeaways

- 预训练 = 自监督的「下一个 token 预测」,CE loss shift by 1
- **8 个训练脚本共用一套骨架**:init_distributed → config → model → dataset → AdamW+cosine → DDP → checkpoint
- 余弦 LR:warmup → 峰值 → 余弦衰减
- AMP(bfloat16)省显存加速;DDP 支持多卡;checkpoint 支持断点续训
- `PretrainDataset` 的 label pad → -100 是关键(ignore_index)

> **核心认知**:预训练循环是后续所有训练阶段的基础。ch9(SFT)只是换了 Dataset 和 lr;ch12(DPO)只是换了 loss;ch13(PPO)只是加了 rollout。骨架不变。

- 精简复习版见 [`./pretrain.ipynb`](./pretrain.ipynb)
- 本章习题与解答见 [`./exercise-solutions.ipynb`](./exercise-solutions.ipynb)

下一章:[第 9 章 · 监督微调 SFT](../ch09/01_main-chapter-code/README.md)